# SiteSafe — Fine-tune Gemma 4 E4B on a free Kaggle T4 GPU

**Goal:** End-to-end reproduction of the SiteSafe fine-tuned model on Kaggle's free T4 (16 GB VRAM). The notebook clones the [SiteSafe repo](https://github.com/SamEthanMathew/sitesafe), builds the OSHA SQLite knowledge base, generates an SFT dataset from the Roboflow Construction Site Safety images, fine-tunes Gemma 4 E4B with Unsloth + LoRA, and exports a `q4_k_m` GGUF you can load straight into Ollama.

## Pre-flight (do this in the Kaggle UI before running cells)

1. **Kaggle → Settings → Accelerator → `GPU T4 x1`** (requires phone-verified Kaggle account)
2. **Kaggle → Settings → Internet → ON** (needed to clone GitHub + pull Hugging Face weights)
3. **Right-hand panel → Add Input → search `construction-site-safety-image-dataset-roboflow`** → add `snehilsanyal/construction-site-safety-image-dataset-roboflow` (~2,801 images, free, CC BY 4.0)

## What you get

- Total runtime: **~30 min** end-to-end (200 steps at LoRA rank 16)
- Output: `/kaggle/working/sitesafe-gemma4-e4b/sitesafe-gemma4-e4b.Q4_K_M.gguf` (≈ 3 GB)
- Download via **right-hand panel → Output → ⋯ → Download**, drop next to `training/Modelfile` locally, then `ollama create sitesafe -f training/Modelfile`

---

## 1. Install dependencies

Unsloth provides 2× faster training and ~70 % less VRAM than vanilla HuggingFace + LoRA. Day-zero support for Gemma 4.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets pillow

## 2. Clone the SiteSafe repo

We pull `data/build_osha_db.py` and `data/build_training_data.py` straight from GitHub so this notebook stays in sync with whatever's on `main`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/SamEthanMathew/sitesafe.git'
REPO_DIR = Path('/kaggle/working/sitesafe')

if REPO_DIR.exists():
    print(f'Repo already cloned at {REPO_DIR}')
else:
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    print(f'Cloned SiteSafe → {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
!ls

## 3. Build the OSHA knowledge base + SFT dataset

**3a.** Build the SQLite KB (63 OSHA construction standards, 12 Subparts).

**3b.** Auto-detect the Roboflow dataset's image / label dirs (Kaggle datasets vary slightly in their on-disk layout).

**3c.** Generate `train.jsonl` — image paths in the JSONL are absolute Kaggle paths, so the trainer can load them directly.

In [ ]:
# 3a — build OSHA DB
!python data/build_osha_db.py

In [ ]:
# 3b — locate the attached construction-safety dataset
#
# We don't hardcode a single mount path. Kaggle has mounted this same slug
# under multiple directory layouts in the wild:
#   /kaggle/input/<slug-derived>/...
#   /kaggle/input/datasets/<owner>/<slug>/...
# AND some uploads of the Roboflow dataset strip the data.yaml — so we fall
# back to a known-good classes list when the YAML is absent.
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')

print('=== /kaggle/input tree (depth=4) ===')
def _tree(p: Path, depth: int = 4, prefix: str = ''):
    if depth < 0 or not p.exists():
        return
    entries = sorted(p.iterdir())[:60]
    for entry in entries:
        print(f'{prefix}{entry.name}{"/" if entry.is_dir() else ""}')
        if entry.is_dir():
            _tree(entry, depth - 1, prefix + '  ')
_tree(INPUT_ROOT)
print('=== end tree ===')

if not INPUT_ROOT.exists() or not list(INPUT_ROOT.iterdir()):
    raise SystemExit(
        'No datasets attached. In the right-hand panel: Add Input -> '
        "search 'construction-site-safety-image-dataset-roboflow' (by snehilsanyal) and add it."
    )

# Step 1: find the train images dir.
img_dir_candidates = []
for parent in INPUT_ROOT.rglob('train'):
    if not parent.is_dir():
        continue
    for sub in ('images', '.'):
        cand = parent / sub if sub != '.' else parent
        if cand.is_dir() and any(cand.glob('*.jpg')):
            img_dir_candidates.append(cand)
# Also check 'images/train' (alternate layout)
for parent in INPUT_ROOT.rglob('images'):
    if not parent.is_dir():
        continue
    cand = parent / 'train'
    if cand.is_dir() and any(cand.glob('*.jpg')):
        img_dir_candidates.append(cand)

print(f'image dir candidates: {[str(p) for p in img_dir_candidates]}')
if not img_dir_candidates:
    raise SystemExit(
        'Could not find a directory of training JPEGs under /kaggle/input/. '
        'See the tree above to see what was attached.'
    )

IMAGES_DIR = img_dir_candidates[0]

# Step 2: find a labels dir near the images.
label_dir_candidates = [
    IMAGES_DIR.parent / 'labels',
    IMAGES_DIR.parent.parent / 'labels' / 'train',
    IMAGES_DIR,  # Roboflow YOLO sometimes keeps labels alongside images
]
LABELS_DIR = next((p for p in label_dir_candidates if p.is_dir() and any(p.glob('*.txt'))), None)
if LABELS_DIR is None:
    raise SystemExit(
        f'Could not find a labels dir near {IMAGES_DIR}. '
        f'Tried: {[str(p) for p in label_dir_candidates]}'
    )

# Step 3: find or synthesize a classes file.
yaml_files = list(INPUT_ROOT.rglob('data.yaml'))
classes_txt_files = list(INPUT_ROOT.rglob('classes.txt')) + list(INPUT_ROOT.rglob('_classes.txt'))
print(f'data.yaml candidates:    {[str(p) for p in yaml_files]}')
print(f'classes.txt candidates:  {[str(p) for p in classes_txt_files]}')

if yaml_files:
    CLASSES_FILE = yaml_files[0]
elif classes_txt_files:
    CLASSES_FILE = classes_txt_files[0]
else:
    # Roboflow Construction Site Safety dataset (snehilsanyal/...) — known class list.
    # Order matches the YOLO label class IDs (0-9) for this dataset.
    KNOWN_CLASSES = [
        'Hardhat',
        'Mask',
        'NO-Hardhat',
        'NO-Mask',
        'NO-Safety Vest',
        'Person',
        'Safety Cone',
        'Safety Vest',
        'machinery',
        'vehicle',
    ]
    CLASSES_FILE = WORKING_ROOT / 'sitesafe-classes.txt'
    CLASSES_FILE.write_text('\n'.join(KNOWN_CLASSES) + '\n', encoding='utf-8')
    print(f'No data.yaml / classes.txt — wrote synthetic class list to {CLASSES_FILE}')

ROBOFLOW_ROOT = IMAGES_DIR.parent.parent

print('ROBOFLOW_ROOT =', ROBOFLOW_ROOT)
print('IMAGES_DIR    =', IMAGES_DIR)
print('LABELS_DIR    =', LABELS_DIR)
print('CLASSES_FILE  =', CLASSES_FILE)
print('image count   =', len(list(IMAGES_DIR.glob('*.jpg'))))
print('label count   =', len(list(LABELS_DIR.glob('*.txt'))))

In [ ]:
# 3c — generate the SFT JSONL from the Roboflow detections
!python data/build_training_data.py \
    --mode roboflow \
    --images-dir {IMAGES_DIR} \
    --labels-dir {LABELS_DIR} \
    --classes-file {CLASSES_FILE} \
    --output /kaggle/working/train.jsonl \
    --compliant-ratio 0.20

!head -1 /kaggle/working/train.jsonl | head -c 400 ; echo
!wc -l /kaggle/working/train.jsonl

## 4. Load Gemma 4 E4B with FastVisionModel

4-bit QLoRA so the 4B model fits with room for vision LoRA on a T4.

In [ ]:
from unsloth import FastVisionModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/gemma-4-E4B-it'

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
print('Model loaded. CUDA available:', torch.cuda.is_available())

## 5. Apply LoRA adapters

If the next cell OOMs, flip `finetune_vision_layers=False` and rerun — text-only LoRA still lifts SiteSafe quality and halves memory.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,    # ← set False if you OOM on T4
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    target_modules='all-linear',
)
model.print_trainable_parameters()

## 6. Load the SFT dataset we just built

In [ ]:
from datasets import Dataset
import json
from pathlib import Path

TRAIN_JSONL = Path('/kaggle/working/train.jsonl')
rows = [json.loads(l) for l in TRAIN_JSONL.read_text().splitlines() if l.strip()]
print(f'Loaded {len(rows)} examples')
assert rows, 'Empty SFT dataset — check Roboflow class mapping in build_training_data.py.'
dataset = Dataset.from_list(rows)
dataset

## 7. Train

200 steps × effective batch 8 (per-device 1 × grad-accum 8) ≈ 1,600 examples seen. Bump `max_steps` to 400–600 if your dataset is large enough to support it without overfitting.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir='/kaggle/working/sitesafe-checkpoints',
        seed=42,
        report_to='none',
    ),
)
trainer.train()

## 8. Save merged model + export GGUF for Ollama

In [ ]:
MERGED_DIR = '/kaggle/working/sitesafe-merged'
model.save_pretrained_merged(MERGED_DIR, tokenizer)
print('Merged checkpoint:', MERGED_DIR)

In [ ]:
model.save_pretrained_gguf(
    '/kaggle/working/sitesafe-gemma4-e4b',
    tokenizer,
    quantization_method='q4_k_m',
)
!ls -lh /kaggle/working/sitesafe-gemma4-e4b/

## 9. Download + register with Ollama (locally)

1. Click **Output** in the right-hand Kaggle panel and download the `.gguf` file.
2. Drop it next to `training/Modelfile` in your local clone of the SiteSafe repo (or update the `FROM` line to point at wherever you saved it).
3. Register it with Ollama:

```bash
ollama create sitesafe -f training/Modelfile
ollama run sitesafe
```

4. Launch the SiteSafe app:

```bash
python app/sitesafe_app.py
```

Open <http://localhost:7860> and upload a construction-site photo.

**Done — you have an OSHA inspector running entirely on your laptop.**